In [1]:
# ============================================================
# 📋 MEETING DECISION EXTRACTOR
# Text and Speech Analysis / NLP Application
# Google Colab - Single Cell
# ============================================================

!pip -q install gradio

import gradio as gr
import re


# ============================================================
# SAMPLE KEYWORDS
# ============================================================

DECISION_WORDS = [
    "decided",
    "decision",
    "agreed",
    "approved",
    "confirmed",
    "finalized",
    "will use",
    "will choose",
    "selected"
]

ACTION_WORDS = [
    "will",
    "must",
    "need to",
    "should",
    "assigned",
    "responsible",
    "complete",
    "prepare",
    "send",
    "create",
    "update",
    "submit",
    "contact",
    "review"
]

DEADLINE_WORDS = [
    "today",
    "tomorrow",
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "saturday",
    "sunday",
    "next week",
    "this week",
    "by",
    "before",
    "deadline"
]


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ============================================================
# SPLIT INTO SENTENCES
# ============================================================

def split_sentences(text):

    sentences = re.split(
        r'(?<=[.!?])\s+|\n+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


# ============================================================
# EXTRACT DECISIONS
# ============================================================

def extract_decisions(sentences):

    decisions = []

    for sentence in sentences:

        lower = sentence.lower()

        for keyword in DECISION_WORDS:

            if keyword in lower:

                if sentence not in decisions:
                    decisions.append(sentence)

                break

    return decisions


# ============================================================
# EXTRACT ACTION ITEMS
# ============================================================

def extract_actions(sentences):

    actions = []

    for sentence in sentences:

        lower = sentence.lower()

        # Avoid adding pure decision sentences as actions
        has_action = any(
            keyword in lower
            for keyword in ACTION_WORDS
        )

        if has_action:

            if sentence not in actions:
                actions.append(sentence)

    return actions


# ============================================================
# EXTRACT DEADLINES
# ============================================================

def extract_deadlines(sentences):

    deadline_items = []

    for sentence in sentences:

        lower = sentence.lower()

        found = []

        for keyword in DEADLINE_WORDS:

            if keyword in lower:
                found.append(keyword)

        if found:

            deadline_items.append(
                (sentence, found)
            )

    return deadline_items


# ============================================================
# EXTRACT PEOPLE
# ============================================================

def extract_people(text):

    # Looks for common meeting patterns:
    # John will...
    # Priya is responsible...
    # Assigned to Rahul...

    patterns = [
        r'\b([A-Z][a-z]+)\s+(?:will|must|should|needs to|is responsible)',
        r'(?:assigned to|responsible to)\s+([A-Z][a-z]+)',
        r'\b([A-Z][a-z]+)\s+is assigned'
    ]

    people = []

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text
        )

        for person in matches:

            if person not in people:
                people.append(person)

    return people


# ============================================================
# MAIN ANALYSIS
# ============================================================

def analyze_meeting(meeting_text):

    if not meeting_text or not meeting_text.strip():

        return (
            "⚠️ Please enter meeting text.",
            "",
            "",
            ""
        )

    text = clean_text(meeting_text)

    sentences = split_sentences(text)

    decisions = extract_decisions(sentences)

    actions = extract_actions(sentences)

    deadlines = extract_deadlines(sentences)

    people = extract_people(text)


    # ========================================================
    # DECISION OUTPUT
    # ========================================================

    decision_output = "📌 DECISIONS\n\n"

    if decisions:

        for i, decision in enumerate(
            decisions, 1
        ):

            decision_output += (
                f"{i}. {decision}\n"
            )

    else:

        decision_output += (
            "No clear decisions detected."
        )


    # ========================================================
    # ACTION OUTPUT
    # ========================================================

    action_output = "✅ ACTION ITEMS\n\n"

    if actions:

        for i, action in enumerate(
            actions, 1
        ):

            action_output += (
                f"{i}. {action}\n"
            )

    else:

        action_output += (
            "No clear action items detected."
        )


    # ========================================================
    # DEADLINE OUTPUT
    # ========================================================

    deadline_output = "⏰ DEADLINES\n\n"

    if deadlines:

        for i, (sentence, found) in enumerate(
            deadlines, 1
        ):

            deadline_output += (
                f"{i}. {sentence}\n"
                f"   Detected: {', '.join(found)}\n\n"
            )

    else:

        deadline_output += (
            "No deadlines detected."
        )


    # ========================================================
    # PEOPLE OUTPUT
    # ========================================================

    people_output = "👥 RESPONSIBLE PEOPLE\n\n"

    if people:

        for i, person in enumerate(
            people, 1
        ):

            people_output += (
                f"{i}. {person}\n"
            )

    else:

        people_output += (
            "No responsible people detected."
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    summary = f"""
📋 MEETING ANALYSIS SUMMARY

📝 Total Sentences: {len(sentences)}

📌 Decisions Found: {len(decisions)}

✅ Action Items Found: {len(actions)}

⏰ Deadline References: {len(deadlines)}

👥 People Detected: {len(people)}
"""

    return (
        summary,
        decision_output,
        action_output,
        deadline_output + "\n" + people_output
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Meeting Decision Extractor"
) as app:

    gr.Markdown(
        """
        # 📋 Meeting Decision Extractor

        ### Automatically extract important information from meeting text.

        **Meeting Text → NLP Analysis → Decisions + Actions + Deadlines + People**
        """
    )

    gr.Markdown(
        """
        ### 🔍 What this application extracts

        📌 Decisions
        ✅ Action Items
        ⏰ Deadlines
        👥 Responsible People
        📊 Meeting Statistics
        """
    )


    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    meeting_input = gr.Textbox(
        label="📝 Enter Meeting Transcript",
        placeholder=(
            "Paste your meeting transcript here..."
        ),
        lines=15
    )


    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    analyze_button = gr.Button(
        "🔍 Extract Meeting Information",
        variant="primary"
    )


    # --------------------------------------------------------
    # OUTPUTS
    # --------------------------------------------------------

    summary_output = gr.Textbox(
        label="📊 Meeting Summary",
        lines=8
    )

    decisions_output = gr.Textbox(
        label="📌 Decisions",
        lines=10
    )

    actions_output = gr.Textbox(
        label="✅ Action Items",
        lines=10
    )

    additional_output = gr.Textbox(
        label="⏰ Deadlines & 👥 People",
        lines=15
    )


    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    analyze_button.click(
        fn=analyze_meeting,
        inputs=meeting_input,
        outputs=[
            summary_output,
            decisions_output,
            actions_output,
            additional_output
        ]
    )


    # --------------------------------------------------------
    # EXAMPLE
    # --------------------------------------------------------

    gr.Examples(
        examples=[
            [
                """The team discussed the new college website.
The team decided to use React for the frontend.
Priya will prepare the homepage design by Friday.
Rahul is responsible for creating the database.
The team agreed to complete the first version next week.
Anita will send the project report tomorrow."""
            ]
        ],
        inputs=meeting_input
    )


    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.Markdown(
        """
        ---

        ### ✨ Features

        📝 Meeting transcript analysis
        📌 Decision extraction
        ✅ Action-item extraction
        ⏰ Deadline detection
        👥 Person detection
        📊 Meeting statistics
        🌐 Interactive web interface

        **Text and Speech Analysis Project**
        """
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92809f86ee0f825b24.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
